# CineIQ — Rating-Based Quality SignalDerives a per-movie quality signal from rating distributions.**Naming note:** This is NOT sentiment analysis. MovieLens has no review text.We derive a quality signal from the numerical ratings themselves:- Ratings >= 4.0 → Positive (+1)- Ratings = 3.0 → Neutral (0)- Ratings <= 2.0 → Negative (-1)Per-movie aggregate is weighted by log(review count) for confidence.

In [ ]:
import pandas as pd
import numpy as np
import pickle

PROC = "../data/processed"
MODELS = "../models"

ratings = pd.read_csv(f"{PROC}/merged.csv")
movies = pd.read_csv(f"{PROC}/movies.csv")
print(f"Ratings: {len(ratings):,}")

## 1. Derive Quality Signal from Ratings

In [ ]:
# Convert ratings to quality signal
# >= 4.0 = positive, = 3.0 = neutral, <= 2.0 = negative
def rating_to_signal(rating):
    if rating >= 4.0:
        return 1.0
    elif rating <= 2.0:
        return -1.0
    else:
        return 0.0

ratings["signal"] = ratings["rating"].apply(rating_to_signal)

In [ ]:
# Aggregate per movie
movie_quality = ratings.groupby("movieId").agg(
    avg_signal=("signal", "mean"),
    num_ratings=("rating", "count"),
    avg_rating=("rating", "mean")
).reset_index()

# Confidence weight: more reviews = more reliable
movie_quality["weighted_signal"] = (
    movie_quality["avg_signal"] * np.log1p(movie_quality["num_ratings"])
)

# Min-max normalize to [0, 1]
min_s = movie_quality["weighted_signal"].min()
max_s = movie_quality["weighted_signal"].max()
movie_quality["quality_score"] = (
    (movie_quality["weighted_signal"] - min_s) / (max_s - min_s)
)

print(f"Quality signal computed for {len(movie_quality):,} movies")
movie_quality.head(10)

## 2. Analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(movie_quality["quality_score"], bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_title("Quality Score Distribution")
axes[0].set_xlabel("Quality Score")

axes[1].scatter(movie_quality["num_ratings"], movie_quality["quality_score"], alpha=0.3, s=5)
axes[1].set_title("Quality Score vs Number of Ratings")
axes[1].set_xlabel("Number of Ratings")
axes[1].set_ylabel("Quality Score")
axes[1].set_xscale("log")

# Top quality movies
top = movie_quality.sort_values("quality_score", ascending=False).head(10)
top = top.merge(movies[["movieId", "title"]], on="movieId")
axes[2].barh(top["title"], top["quality_score"], color="teal", alpha=0.8)
axes[2].set_title("Top 10 Quality Movies")
axes[2].set_xlabel("Quality Score")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

## 3. Save

In [ ]:
# Save quality signal
movie_quality.to_csv(f"{PROC}/movie_quality_signal.csv", index=False)
print(f"Saved movie_quality_signal.csv ({len(movie_quality):,} movies)")

# Verify
loaded = pd.read_csv(f"{PROC}/movie_quality_signal.csv")
print(f"Columns: {loaded.columns.tolist()}")
print(f"Shape: {loaded.shape}")